# Realistic avatar from a full-body photo

Rebuilds the person in 3D from a single photo, re-renders them from a **different camera height**,
and finishes the render with SDXL so it reads as a studio photograph.

The body is reconstructed geometrically rather than re-imagined by a generative model, so real
proportions are preserved: a heavy body stays heavy. The diffusion pass runs at low denoising
strength under depth control purely to add photographic texture.

**Runtime → Change runtime type → GPU** before running. A T4 is enough.

## 1. Environment

In [ ]:
!nvidia-smi -L || echo "NO GPU - set Runtime > Change runtime type > GPU"

In [ ]:
# Colab already has a working torch; only the rest is installed here.
!pip install -q "transformers>=4.45" "diffusers>=0.31" "accelerate>=0.34" \
                "safetensors>=0.4" "huggingface_hub>=0.25" "opencv-python-headless>=4.9"

In [ ]:
!git clone -q -b claude/cool-clarke-jsux4r https://github.com/mwmilad/generation_avatar.git
%cd generation_avatar

## 2. Upload a full-body photo

Works best with: the whole body in frame including the feet, the person roughly facing the camera,
and a reasonably plain background.

In [ ]:
from google.colab import files

uploaded = files.upload()
SOURCE = next(iter(uploaded))
print("using", SOURCE)

## 3. Geometry preview (fast)

Runs segmentation, depth and the 3D re-render but skips diffusion, so it takes seconds instead of
minutes. Use it to dial in the camera before paying for the refinement pass.

Watch the printed **source camera height** - that is the height your photo was taken from. Pick a
target below it to lower the camera.

In [ ]:
from avatar import PipelineConfig, generate_avatar

cfg = PipelineConfig()

# --- the person -----------------------------------------------------------
cfg.camera.subject_height_m = 1.70      # real height, metres. Sets the whole scale.

# --- the camera that took the photo ---------------------------------------
cfg.camera.focal_mm_equiv = 28.0        # 26-28 for a phone, 50-85 for a portrait lens

# --- the camera you want instead ------------------------------------------
cfg.camera.target_camera_height_m = 1.15   # lower than the source height = looking up
cfg.camera.target_distance_m = None        # None keeps the original distance.
                                           # Raising it flattens perspective - distance is what
                                           # controls that, not the lens below.
cfg.camera.orbit_deg = 0.0                 # rotate around the subject
cfg.camera.aim_height_frac = 0.55          # 0.5 waist, 0.9 head
cfg.camera.out_focal_mm_equiv = None       # None fits the framing to the subject

# --- output ---------------------------------------------------------------
cfg.render.width, cfg.render.height = 896, 1216
cfg.render.background = "#f2f0ee"

cfg.refine.enabled = False               # preview only
preview = generate_avatar(SOURCE, cfg, "output")

In [ ]:
from IPython.display import display
from PIL import Image

for name in ("mask", "depth", "render"):
    if name in preview:
        print(name)
        display(Image.open(preview[name]).resize((320, 434)))

### If the preview looks wrong

| symptom | fix |
|---|---|
| body too wide or too thin front-to-back | `cfg.geometry.back_shell_scale` (default 0.9) |
| perspective change too strong / too weak | `cfg.camera.focal_mm_equiv` - the source lens guess |
| subject cropped or too small | `cfg.camera.aim_height_frac`, `cfg.camera.target_distance_m` |
| large "unseen surface" percentage | move the camera less, or raise `cfg.geometry.max_point_upsample` |
| halo or fringe around the outline | `cfg.geometry.mask_erode_px` |
| outline not following the body | set `cfg.segmenter = "briaai/RMBG-1.4"` for a finer matte |

## 4. Full render with refinement

Downloads SDXL and the depth ControlNet on the first run (~7 GB) and takes a few minutes.

In [ ]:
cfg.refine.enabled = True
cfg.refine.strength = 0.30        # keep low: high strength lets SDXL reshape the body
cfg.refine.face_pass = True
cfg.refine.seed = 0

result = generate_avatar(SOURCE, cfg, "output")
display(Image.open(result["avatar"]))

In [ ]:
files.download(str(result["avatar"]))

## 5. Notes on fidelity

`cfg.refine.strength` is the dial that trades texture quality against faithfulness. Everything above
about **0.45** starts letting the model reshape the silhouette toward its training average, which is
what makes most avatar pipelines quietly slim people down. If the output looks plastic, prefer
raising `cfg.refine.steps` or improving the source photo over raising strength.

Regions the original photo could never see - the underside of the chin, chest and belly that a lower
camera reveals - are genuinely reconstructed, not observed. The pipeline reports how much of the
frame that is, and refines those areas harder via `cfg.refine.novel_strength`.